In [86]:
import warnings
warnings.filterwarnings('ignore')

In [87]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rapidfuzz import fuzz, process

In [88]:
url = "https://www.nytimes.com/newsgraphics/polls/house.csv"
polls = pd.read_csv(url)
polls.to_csv('../../2026_data/polls/house.csv')
polls = polls[
    (polls['state'] != 'US') &
    (polls['stage'] == 'general')
]
polls.head()

,poll_id,pollster_id,pollster,sponsor_ids,sponsors,display_name,pollster_rating_id,pollster_rating_name,numeric_grade,pollscore,methodology,transparency_score,state,start_date,end_date,sponsor_candidate_id,sponsor_candidate,sponsor_candidate_party,question_id,sample_size,population,subpopulation,population_full,tracking,created_at,notes,url,url_article,url_topline,url_crosstab,source,internal,partisan,cycle,office_type,seat_name,seat_number,election_date,stage,party,pct,answer,candidate_name,candidate_id,race_id,ranked_choice_round,ranked_choice_reallocated,ranked_choice_final,nationwide_match,hypothetical
7,491431af-a6d9-4569-b686-06c50a069cb0,508ef336-8100-45cc-aec5-d506b2a053b8,Impact Research,20c85a44-ad67-4d54-82bd-b78cdcaf9ad8,House Majority PAC,Impact Research,14.0,Impact Research,NaN,NaN,NaN,NaN,FL,8/24/26,8/31/26,NaN,NaN,NaN,59067f58-f491-4f4f-bf15-aa13f8ea2665,NaN,lv,NaN,lv,NaN,9/9/26 10:38,NaN,https://www.thehousemajoritypac.com/news/hmp-p...,NaN,NaN,NaN,NaN,NaN,DEM,2026,U.S. House,Twenty-second Congressional District,22.0,2026-11-03,general,REP,46.0,Republican,Generic Republican,97a3a951-7de9-49d7-b305-e6f1797d0217,0fcc8534-599c-4827-8c00-6875ee27305f,NaN,False,False,NaN,NaN
8,491431af-a6d9-4569-b686-06c50a069cb0,508ef336-8100-45cc-aec5-d506b2a053b8,Impact Research,20c85a44-ad67-4d54-82bd-b78cdcaf9ad8,House Majority PAC,Impact Research,14.0,Impact Research,NaN,NaN,NaN,NaN,FL,8/24/26,8/31/26,NaN,NaN,NaN,59067f58-f491-4f4f-bf15-aa13f8ea2665,NaN,lv,NaN,lv,NaN,9/9/26 10:38,NaN,https://www.thehousemajoritypac.com/news/hmp-p...,NaN,NaN,NaN,NaN,NaN,DEM,2026,U.S. House,Twenty-second Congressional District,22.0,2026-11-03,general,DEM,46.0,Democrat,Generic Democrat,d73e9669-a0c1-41c3-b768-9e0355cf5fec,0fcc8534-599c-4827-8c00-6875ee27305f,NaN,False,False,NaN,NaN
9,491431af-a6d9-4569-b686-06c50a069cb0,508ef336-8100-45cc-aec5-d506b2a053b8,Impact Research,20c85a44-ad67-4d54-82bd-b78cdcaf9ad8,House Majority PAC,Impact Research,14.0,Impact Research,NaN,NaN,NaN,NaN,FL,8/24/26,8/31/26,NaN,NaN,NaN,be32d0ec-f1d5-4656-aa56-7494d1f5b197,NaN,lv,NaN,lv,NaN,9/9/26 10:38,NaN,https://www.thehousemajoritypac.com/news/hmp-p...,NaN,NaN,NaN,NaN,NaN,DEM,2026,U.S. House,Twenty-second Congressional District,22.0,2026-11-03,general,REP,44.0,Askar,Casey Askar,c410a0a8-0a03-4ef5-ba01-51c4d353da0e,0fcc8534-599c-4827-8c00-6875ee27305f,NaN,False,False,NaN,NaN
10,491431af-a6d9-4569-b686-06c50a069cb0,508ef336-8100-45cc-aec5-d506b2a053b8,Impact Research,20c85a44-ad67-4d54-82bd-b78cdcaf9ad8,House Majority PAC,Impact Research,14.0,Impact Research,NaN,NaN,NaN,NaN,FL,8/24/26,8/31/26,NaN,NaN,NaN,be32d0ec-f1d5-4656-aa56-7494d1f5b197,NaN,lv,NaN,lv,NaN,9/9/26 10:38,NaN,https://www.thehousemajoritypac.com/news/hmp-p...,NaN,NaN,NaN,NaN,NaN,DEM,2026,U.S. House,Twenty-second Congressional District,22.0,2026-11-03,general,DEM,46.0,Dandiya,Pia Dandiya,86247ed9-41be-4202-aa67-30a67d7d8706,0fcc8534-599c-4827-8c00-6875ee27305f,NaN,False,False,NaN,NaN
11,1ecb97e5-f461-4791-b993-d7ec2454c20c,433ef9e7-7784-4272-92fc-ad8ae9efb109,Tulchin Research,20c85a44-ad67-4d54-82bd-b78cdcaf9ad8,House Majority PAC,Tulchin Research,340.0,Tulchin Research,NaN,NaN,NaN,NaN,AL,8/24/26,8/31/26,NaN,NaN,NaN,dbf87c16-e121-44ee-82cb-87167a927941,NaN,lv,NaN,lv,NaN,9/9/26 10:37,NaN,https://www.thehousemajoritypac.com/news/hmp-p...,NaN,NaN,NaN,NaN,NaN,DEM,2026,U.S. House,Second Congressional District,2.0,2026-11-03,general,REP,47.0,Marques,Rhett Marques,f8bc7ef2-d7de-4874-b548-5801205661c5,c5e7e34c-92d5-42b6-ad57-e5c784dcdf20,NaN,False,False,NaN,NaN


In [89]:
pd.set_option('display.max_columns', None)

In [90]:
polls_pivot = pd.pivot_table(data=polls, values='pct', index=['poll_id', 'question_id', 'state', 'seat_number'], columns='candidate_name', aggfunc='first')
polls_pivot = polls_pivot.reset_index()
polls_pivot.head()

candidate_name,poll_id,question_id,state,seat_number,Aaron Flint,Adam Ortiz,Alex Kelloff,Alexander Fornino,Amanda Green,Ammar Campa-Najjar,Andy Harris,Andy Ogles,Ann Wagner,Anna Paulina Luna,Anthony Constantino,Ashley Bell,Austin Ahlman,Bajun Mavalwalla,Bale Dalton,Becca Balint,Belinda Keiser,Bill Hill,Bill Huizenga,Bill Reeside,Blake Gendebien,Bob Brooks,Bob Harvie,Bobby Pulido,Brad Finstad,Brad Smith,Brandon Herrera,Brendan Gomez,Brennan Barrington,Brian Fitzpatrick,Brian Poindexter,Brice Barnes,Bridget Brink,Bryan Steil,Cait Conley,Carlos De La Cruz,Carlos Gimenez,Carmela Conroy,Casey Askar,Charlie Hatcher,Chaz Molder,Cherlynn Stevenson,Chris Backemeyer,Chris Gallant,Chris Getty,Chris Jones,Chris Royal,Chris Schmidt,Christina Bohannan,Christina Hines,Clay Strickland,Connie Chan,Cory Mills,Curtis Goodwin,Dan Green,Dan Schwartz,Daniella Levine Cava,Darrell Issa,Darren Soto,David Flippo,David G. Valadao,David Gedert,David Rouzer,David Womack,Denise Powell,Derek Merrin,Derrick Van Orden,Don Davis,Don Leonard,Don't know,Don't know/Someone else,Don't know/Would not vote,Doris Matsui,Dwayne Romero,Ed Hershey,Eddie Espinoza,Elaine G. Luria,Eli Crane,Eliott Rodriguez,Eric Chung,Eric Conroy,Eric Flores,Eric Hafner,Eric Pratt,Esther Kim-Varet,Fred Wellman,French Hill,Gabe Vasquez,Generic Democrat,Generic Libertarian,Generic Republican,George Moraitis,Gerald Malloy,Gina Swoboda,Glenn Grothman,Greg Cunningham,Greg Landsman,Greg Murphy,Hector Mujica,Heidi Hall,Henry Cuellar,J.D. Ford,Jackie Auringer,Jake Johnson,James Johnson,James Pericola,Janelle Stelson,Jared Golden,Jared Moskowitz,Jasmeet Bains,Jay Feely,Jeff Crank,Jeff Hurd,Jeff Van Drew,Jen Kiggans,Jennifer Jenkins,Jennifer Konfrst,Jenny Costa Honeycutt,Jessica Killin,Jim Desmond,JoAnna Mendoza,Joe Baldacci,Joe Mitchell,Joe Strada,John Braun,John Cavanaugh,John McGuire,John Vincent,John Williams,Johnny Garcia,Jonathan Nez,Jordan Wood,Juan Ciscomani,Kaela Berg,Kathy Castor,Katy Padilla Stout,Kaylee Peterson,Keith Gross,Kelly Kirschner,Ken Calvert,Kevin Kiley,Kevin Steele,Kimberly Hardy,Kristina Knickerbocker,LaKesha Womack,Laurie Buckhout,Leela Gray,Lily Tang Williams,Lindsay James,Lupe Castillo,Lynn Chapman,Maad Abu-Ghazalah,Mac Deford,Maggie Goodlander,Mai Vang,Marcy Kaptur,Margo Ellis,Mariannette Miller-Meeks,Marie Gluesenkamp Perez,Mark Coester,Mark Smith,Marlon Duran,Marni von Wilpert,Mary Peltola,María Elvira Salazar,Matt Cavanaugh,Matt Dunlap,Matt Klein,Matt Little,Matt Maasdam,Matt Rains,Matt Schultz,Max Miller,Michael Baumgartner,Michael Bridgford,Michael Thurow,Micheál O'Leary,Mike Beltran,Mike Bouchard,Mike Carey,Mike Davey,Mike Flood,Mike Haridopolos,Mike Lawler,Mike Turner,Mitchell Berman,Monica De La Cruz,Nancy Lacore,Nancy Pelosi,Nate Powell,Neither,Nicholas J. LaLota,Nicholas Zateslo,Nick Begich,Oliver Larkin,Other named candidates,Paige Cognetti,Pat Harrigan,Pat Ryan,Patty Garcia,Paul LePage,Pia Dandiya,Ralph Alvarado,Randy Villegas,Raymond Smith Jr.,Rebecca Bennett,Rebecca Cooke,Rhett Marques,Rich McCormick,Richard Lamondin,Richard Pan,Rob Bresnahan Jr.,Robert Wittman,Robin Peguero,Russ Fulcher,Russell Fry,Ryan Busse,Ryan Dotson,Ryan E. Mackenzie,Ryan Zinke,Saikat Chakrabarti,Sam Forstag,Sarah Trone Garriott,Sarah Zabel,Scott Perry,Scott Singer,Scott Wiener,Sean McCann,Shannon Taylor,Shomari Figures,Someone else,Sydney Gruters,Tano Tijerina,Teresa Benitez-Thompson,Thomas Aaron Bailey,Thomas H. Kean Jr.,Thomas McMasters,Tim Greimel,Tim Moore,Tim Sheehy,Tina Shah,Tom Barrett,Tom Perriello,Tony D’Arrigo,Tony Kozycki,Vicente Gonzalez,Victoria Spartz,William Lawrence,Would not vote,Yen Bailey,Young Kim,Zach Dembo,Zach Nunn
0,016369fb-4a26-42ad-a583-39067289cab2,65c0d75b-5121-4107-b61a-95b2534fab6e,WI,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.0,N

In [91]:
polls_pivot.shape

(275, 257)

In [92]:
candinfo_url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vQZC7cVru6ltLR2e8XN5jdJPfxfj42BAxUApe3Zq3_ENQjLtYntmAxD0pIHqEUJ4ZFLXlybKJdkLf2r/pub?output=csv'
candinfo = pd.read_csv(candinfo_url)

In [93]:
dem_uncont = candinfo[candinfo['rep_cand'] == 'Not Contested']
rep_uncont = candinfo[candinfo['dem_cand'] == 'Not Contested']
candinfo = candinfo[(candinfo['dem_cand'] != 'Not Contested') &
    (candinfo['rep_cand'] != 'Not Contested')]
candinfo['state_po'] = candinfo['cd'].map(lambda x: x[:2]).astype(str)
candinfo['district_number'] = candinfo['cd'].map(lambda x: 0 if x[3:] == 'AL' else int(x[3:]))
candinfo.head()

,cd,dem_cand,rep_cand,dem_inc_any,rep_inc_any,state_po,district_number
0,AK-AL,Bill Hill,Nick Begich,False,True,AK,0
1,AL-01,Clyde Jones Jr.,Jerry Carl,False,False,AL,1
2,AL-02,Shomari Figures,Rhett Marques,True,False,AL,2
3,AL-03,Lee McInnis,Mike Rogers,False,True,AL,3
4,AL-04,Amanda Pusczek,Robert Aderholt,False,True,AL,4


In [94]:
dem_uncont.shape, rep_uncont.shape

((16, 5), (1, 5))

In [95]:
polled_cands = polls_pivot.columns.values[4:]
running_dems = np.unique(candinfo['dem_cand'].astype(str))
running_reps = np.unique(candinfo['rep_cand'].astype(str))
all_running_cands = np.concatenate([running_dems, running_reps])

In [96]:
hypo_cands = []
for c in polled_cands:
    fuzzymatch = process.extractOne(c, all_running_cands, scorer=fuzz.token_sort_ratio, score_cutoff=80)
    if fuzzymatch is None:
        hypo_cands.append(c)

In [97]:
for h in hypo_cands:
    if h in ['Nicholas J. LaLota', 'Robert J. Wittman', 'Thomas H. Kean',
 'Thomas H. Kean Jr.', "Don't know",
 "Don't know/Someone else",
 "Don't know/Would not vote", 'Someone else', 'Would not vote']:
        continue
    # print(h, polls_pivot["Christina Bohannan"].isna().all())
    polls_pivot = polls_pivot[polls_pivot[h].isna()]
    polls_pivot = polls_pivot.drop([h], axis=1)

for h in ['John Williams', 'Mike Davey']: # names that weren't caught in hypo_cands but should've been
    polls_pivot = polls_pivot[polls_pivot[h].isna()]
    polls_pivot = polls_pivot.drop([h], axis=1)

In [98]:
polls_pivot.shape

(126, 171)

In [99]:
rel_polls = polls[(polls['poll_id'].isin(np.unique(polls_pivot['poll_id']))) &
    (polls['question_id'].isin(np.unique(polls_pivot['question_id'])))]

In [100]:
np.unique(rel_polls['candidate_name'])

array(['Aaron Flint', 'Andy Harris', 'Ann Wagner', 'Anna Paulina Luna',
       'Anthony Constantino', 'Ashley Bell', 'Bill Hill', 'Bill Huizenga',
       'Blake Gendebien', 'Bob Brooks', 'Bob Harvie', 'Bobby Pulido',
       'Brad Finstad', 'Brad Smith', 'Brandon Herrera',
       'Brian Fitzpatrick', 'Brian Poindexter', 'Bryan Steil',
       'Cait Conley', 'Carlos De La Cruz', 'Carmela Conroy',
       'Casey Askar', 'Chris Backemeyer', 'Chris Gallant', 'Chris Jones',
       'Chris Schmidt', 'Christina Bohannan', 'Christina Hines',
       'Dan Green', 'Dan Schwartz', 'Darren Soto', 'David Flippo',
       'David G. Valadao', 'David Rouzer', 'Derrick Van Orden',
       'Don Davis', "Don't know", "Don't know/Someone else",
       "Don't know/Would not vote", 'Dwayne Romero', 'Elaine G. Luria',
       'Eli Crane', 'Eliott Rodriguez', 'Eric Chung', 'Eric Conroy',
       'Eric Flores', 'Eric Pratt', 'Fred Wellman', 'French Hill',
       'Gabe Vasquez', 'Glenn Grothman', 'Greg Cunningham',
    

In [101]:
rel_polls.shape

(342, 50)

In [102]:
rel_polls.to_csv('transformed/relevent_house_polls.csv')